# GNINA Docking Workflow

This notebook performs molecular docking using [GNINA](https://github.com/gnina/gnina), a deep-learning-augmented molecular docking program, to generate binding pose candidates for a protein–ligand complex.

## Workflow overview
1. Run GNINA docking (10 poses per ligand, whole-protein autobox)
2. Extract top-10 poses by CNN pose score, save as individual SDF files
3. Add hydrogens at pH 7.4 (OpenBabel)
4. Calculate RMSD to crystal/reference structure (if available)

## Prerequisites
- [GNINA](https://github.com/gnina/gnina) installed and on `PATH`
- [OpenBabel](https://openbabel.org/) (`obabel`) installed and on `PATH`
- Python packages: `rdkit`, `numpy`, `pandas`, `scipy` (see `environment.yml`)

## Input files required
- `receptor.pdb`: protein structure prepared with a structure-preparation tool (e.g., OpenMM Setup)
- `ligand.sdf`: ligand structure with hydrogens at pH 7.4

## Reference
This workflow was used to generate binding pose candidates for the HSA binding pose ML study.
See the manuscript for details on receptor and ligand preparation.

## 1. Setup

In [ ]:
from pathlib import Path
from rdkit import Chem
import csv
import os
import subprocess
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
import random

## 2. User parameters

Set the following paths and identifiers before running.

In [ ]:
# ── User-defined parameters ─────────────────────────────────────────────────

# Project / complex identifier (e.g., PDB ID)
PROJECT_ID = "1E7A"  # change to your complex ID

# Path to the prepared receptor PDB (all hydrogens added, waters removed)
RECEPTOR_PDB = Path(f"{PROJECT_ID}-processed.pdb")

# Path to the ligand SDF (hydrogens added at pH 7.4)
LIGAND_SDF = Path(f"{PROJECT_ID}_ligand.sdf")

# Directory containing crystal/reference ligand SDF file(s) for RMSD calculation (None to skip)
CRYSTAL_DIR = Path("crystal_ligands")  # set to None to skip RMSD

# Output directory for docking results
OUTPUT_DIR = Path(f"results/{PROJECT_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Number of docking poses to retrieve
N_POSES = 10

# Autobox padding (Angstroms)
AUTOBOX_ADD = 4.0

print(f"Project ID : {PROJECT_ID}")
print(f"Receptor   : {RECEPTOR_PDB}")
print(f"Ligand     : {LIGAND_SDF}")
print(f"Output dir : {OUTPUT_DIR}")

## 3. GNINA docking

Runs GNINA in whole-protein autobox mode (`--autobox_ligand` set to the receptor).
Default settings match the experimental workflow: `--exhaustiveness 64` and `--seed 123`.
CPU-only mode (`--no_gpu`) is used for reproducibility.
Adjust `--num_modes` for the number of output poses.

In [ ]:
docked_sdf = OUTPUT_DIR / "docked.sdf"

# Use half of available CPU threads
cpu_threads = max(1, (os.cpu_count() or 1) // 2)

cmd = [
    "gnina",
    "-r", str(RECEPTOR_PDB),
    "-l", str(LIGAND_SDF),
    "--autobox_ligand", str(RECEPTOR_PDB),  # whole-protein docking
    "--autobox_add", str(AUTOBOX_ADD),
    "--no_gpu",  # for CPU-only
    "--cpu", str(os.cpu_count()),
    "--num_modes", str(N_POSES),
    "--exhaustiveness", "64",
    "--seed", "123",
    "-o", str(docked_sdf),
]

print("Running GNINA:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("GNINA docking failed. Check the error message above.")
print(f"\nDocked output saved to: {docked_sdf}")

## 4. Extract top-10 poses

Splits the docked SDF into individual SDF files (`pose01.sdf` – `pose10.sdf`) and
records GNINA scores (`vina_affinity`, `cnn_pose_score`, `cnn_affinity`) in a CSV.

GNINA outputs poses in descending order of CNN pose score by default, so `pose01`
corresponds to the highest CNN pose score.

In [ ]:
suppl = Chem.SDMolSupplier(str(docked_sdf), removeHs=False)
mols = [m for m in suppl if m is not None]
print(f"Total poses retrieved: {len(mols)}")

records = []
for idx, mol in enumerate(mols, start=1):
    props = {name: mol.GetProp(name) for name in mol.GetPropNames()}
    affinity  = float(props.get("minimizedAffinity", float("nan")))
    cnn_score = float(props.get("CNNscore",          float("nan")))
    cnn_aff   = float(props.get("CNNaffinity",       float("nan")))
    records.append((idx, affinity, cnn_score, cnn_aff, mol))

# Save each pose as an individual SDF (pose01 = highest CNN pose score)
csv_path = OUTPUT_DIR / f"{PROJECT_ID}_gnina_scores.csv"
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["pose_index", "vina_affinity", "cnn_pose_score", "cnn_affinity"])
    for idx, affinity, cnn_score, cnn_aff, mol in records:
        pose_tag = f"pose{idx:02d}"
        sdf_out = OUTPUT_DIR / f"{pose_tag}.sdf"
        writer_sdf = Chem.SDWriter(str(sdf_out))
        writer_sdf.write(mol)
        writer_sdf.close()
        writer.writerow([idx, affinity, cnn_score, cnn_aff])
        print(f"  {pose_tag}: vina_affinity={affinity:.3f}, cnn_pose_score={cnn_score:.3f}")

print(f"\nScores saved to: {csv_path}")

## 5. Add hydrogens at pH 7.4 (OpenBabel)

Re-applies pH-dependent protonation to each pose SDF using OpenBabel.
This overwrites the individual pose SDF files in-place.

In [ ]:
for sdf_file in sorted(OUTPUT_DIR.glob("pose*.sdf")):
    cmd_h = [
        "obabel",
        str(sdf_file),
        "-O", str(sdf_file),
        "-p", "7.4",
    ]
    result = subprocess.run(cmd_h, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"WARNING: obabel failed for {sdf_file.name}: {result.stderr.strip()}")
    else:
        print(f"  pH 7.4 protonation applied to {sdf_file.name}")

## 6. RMSD to crystal structure (optional)

Calculates heavy-atom RMSD between each docked pose and the crystal/reference ligand(s)
using the Hungarian algorithm for atom matching (handles symmetry).
When multiple crystal ligands exist (e.g., multi-ligand complexes), the minimum RMSD
across all references is used. Handles cases where atom counts differ between
reference and docked structures.

Skip this cell if `CRYSTAL_DIR` is `None` or the directory does not exist.

In [ ]:
def get_heavy_atom_coords(mol):
    """Extract heavy-atom coordinates as Nx3 numpy array."""
    conf = mol.GetConformer()
    pts = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 1:
            continue
        pos = conf.GetAtomPosition(atom.GetIdx())
        pts.append([pos.x, pos.y, pos.z])
    return np.array(pts)


def calc_rmsd_hungarian(coords_ref, coords_docked):
    """Heavy-atom RMSD with Hungarian-algorithm atom matching.

    Handles cases where the number of heavy atoms differs between
    reference and docked structures by using partial matching.
    """
    D = np.linalg.norm(coords_ref[:, None, :] - coords_docked[None, :, :], axis=-1)
    row_ind, col_ind = linear_sum_assignment(D)
    return float(np.sqrt((D[row_ind, col_ind] ** 2).sum() / len(row_ind)))


if CRYSTAL_DIR is not None and Path(CRYSTAL_DIR).is_dir():
    # Load all crystal/reference ligands from the directory
    crystal_paths = sorted(Path(CRYSTAL_DIR).glob(f"{PROJECT_ID}_*_*.sdf"))
    crystals = []
    for cp in crystal_paths:
        m = Chem.MolFromMolFile(str(cp), removeHs=True)
        if m and m.GetNumConformers() > 0:
            crystals.append(m)

    if not crystals:
        print(f"WARNING: No valid crystal ligand SDFs found in {CRYSTAL_DIR}. Skipping RMSD.")
    else:
        print(f"Loaded {len(crystals)} crystal ligand(s) for RMSD calculation")
        rmsd_results = []
        for sdf_file in sorted(OUTPUT_DIR.glob("pose*.sdf")):
            mol_d = Chem.MolFromMolFile(str(sdf_file), removeHs=True)
            if mol_d is None or mol_d.GetNumConformers() == 0:
                rmsd_results.append((sdf_file.stem, float("nan")))
                continue

            dock_coords = get_heavy_atom_coords(mol_d)
            best_rmsd = float("inf")
            # Compare against each crystal ligand, keep minimum RMSD
            for cry in crystals:
                cry_coords = get_heavy_atom_coords(cry)
                rmsd = calc_rmsd_hungarian(cry_coords, dock_coords)
                best_rmsd = min(best_rmsd, rmsd)
            rmsd_results.append((sdf_file.stem, best_rmsd))
            print(f"  {sdf_file.stem}: RMSD = {best_rmsd:.3f} Å")

        # Append RMSD to score CSV
        df = pd.read_csv(csv_path)
        rmsd_dict = dict(rmsd_results)
        df["rmsd_to_crystal_A"] = df["pose_index"].apply(lambda x: rmsd_dict.get(f"pose{x:02d}"))
        df.to_csv(csv_path, index=False)
        print(f"\nRMSD values appended to: {csv_path}")
else:
    print("Crystal directory not found or CRYSTAL_DIR=None — skipping RMSD calculation.")

## 7. Summary

After running this notebook:
- `results/<PROJECT_ID>/docked.sdf` — all docked poses (GNINA output)
- `results/<PROJECT_ID>/pose01.sdf` – `pose10.sdf` — individual poses (pH 7.4 protonated; pose01 = highest CNN pose score)
- `results/<PROJECT_ID>/<PROJECT_ID>_gnina_scores.csv` — GNINA scores (and RMSD if crystal provided)

The pose SDF files are used as input for the MD simulation workflow (`notebooks/md_simulation_workflow.ipynb`).